# Milestone 5: From Raw Data to Predictions

This notebook tests different preprocessing methods for the Steam game hit prediction model. We compare standardization, one-hot encoding and TF-IDF, then combine the selected steps into a pipeline.

In [19]:
# Data handling
import pandas as pd
import numpy as np

# Machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import cross_val_score

# Evaluation
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Imports completed successfully!")

Imports completed successfully!


## 1. Load the cleaned datasets

Load the existing training, validation and test datasets created in the earlier milestones.

In [20]:
# Load the existing datasets
train_df = pd.read_parquet("../data/train.parquet")
val_df = pd.read_parquet("../data/validation.parquet")
test_df = pd.read_parquet("../data/test.parquet")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Training: (58516, 45)
Validation: (19506, 45)
Test: (19506, 45)


## 2. Prepare the features

We use information that is available before a game is released. We exclude information that could cause data leakage, such as reviews, estimated owners and peak concurrent players.

In [21]:
# Display all available columns
print("Available columns:")
print(train_df.columns.tolist())

print("\nDataset preview:")
display(train_df.head())

Available columns:
['appID', 'name', 'release_date', 'estimated_owners', 'peak_ccu', 'required_age', 'price', 'dlc_count', 'detailed_description', 'short_description', 'supported_languages', 'full_audio_languages', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'user_score', 'positive', 'negative', 'score_rank', 'achievements', 'recommendations', 'notes', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'developers', 'publishers', 'categories', 'genres', 'tags', 'screenshots', 'movies', 'packages', 'release_year', 'rank_in_year', 'games_in_year', 'hit']

Dataset preview:


,appID,name,release_date,estimated_owners,peak_ccu,required_age,price,dlc_count,detailed_description,short_description,...,categories,genres,tags,screenshots,movies,packages,release_year,rank_in_year,games_in_year,hit
0,2888030,Coyote: An Old West Vignette,2024-05-30,0 - 0,0,0,0.00,0,Coyote: An Old West Vignette is a tiny point-a...,Coyote: An Old West Vignette is a tiny point-a...,...,"[Single-player, Full controller support, Steam...","[Adventure, Indie, Free To Play]",[],[https://shared.akamai.steamstatic.com/store_i...,[],[],2024,17032.0,20031,0
1,753420,Dungreed,2018-02-14,200000 - 500000,123,0,4.99,1,Dungreed is a 2D side-scrolling action game wi...,Dungreed is a 2D side-scrolling action game wi...,...,"[Single-player, Steam Achievements, Full contr...","[Action, Indie]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Dungreed"", ""description"": """", ...",2018,145.0,7461,1
2,747360,Gray Dawn,2018-06-07,50000 - 100000,1,0,1.99,0,You will be entangled in a terrifying adventur...,Embark on a terrifying adventure of a priest a...,...,"[Single-player, Steam Achievements, Full contr...","[Adventure, Indie]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Gray Dawn"", ""description"": """",...",2018,510.0,7461,1
3,2463530,Vacation Paradise: Florida Collector's Edition,2023-07-22,0 - 20000,1,0,6.49,0,Take a Vacation with us to the sunshine state ...,"Take a Vacation in Florida! Beautiful beaches,...",...,"[Single-player, Family Sharing]","[Action, Adventure, Casual, Indie]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Vacation Paradise: Florida Col...",2023,14369.0,14596,0
4,1269540,Slash Roll,2020-08-26,0 - 20000,0,0,2.99,0,Remember to wishlist the game About the Game A...,"After slaying some spire, the taverns of Slash...",...,"[Single-player, Steam Achievements, Steam Clou...","[Indie, Strategy]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Slash Roll"", ""description"": """"...",2020,5637.0,8804,0


## 3. Inspecting candidate features

Before building the preprocessing steps, we inspect the numerical, categorical, list and text columns. We also check missing values to avoid problems when training the models.

In [22]:
# Columns we want to investigate
candidate_columns = [
    "price",
    "required_age",
    "windows",
    "mac",
    "linux",
    "release_date",
    "categories",
    "genres",
    "short_description"
]

# Display data types and missing values
feature_info = pd.DataFrame({
    "Data type": train_df[candidate_columns].dtypes,
    "Missing values": train_df[candidate_columns].isnull().sum()
})

display(feature_info)

# Check example values
for column in candidate_columns:
    print(f"\n{column}:")
    print(train_df[column].head(3).tolist())

,Data type,Missing values
price,float64,0
required_age,int64,0
windows,bool,0
mac,bool,0
linux,bool,0
release_date,datetime64[us],0
categories,object,0
genres,object,0
short_description,str,0



price:
[0.0, 4.99, 1.99]

required_age:
[0, 0, 0]

windows:
[True, True, True]

mac:
[False, True, False]

linux:
[False, True, False]

release_date:
[Timestamp('2024-05-30 00:00:00'), Timestamp('2018-02-14 00:00:00'), Timestamp('2018-06-07 00:00:00')]

categories:
[array(['Single-player', 'Full controller support', 'Steam Cloud'],
      dtype=object), array(['Single-player', 'Steam Achievements', 'Full controller support',
       'Steam Trading Cards', 'Steam Workshop', 'Steam Cloud',
       'Family Sharing'], dtype=object), array(['Single-player', 'Steam Achievements', 'Full controller support',
       'Steam Trading Cards', 'Steam Cloud', 'Remote Play on TV',
       'Family Sharing'], dtype=object)]

genres:
[array(['Adventure', 'Indie', 'Free To Play'], dtype=object), array(['Action', 'Indie'], dtype=object), array(['Adventure', 'Indie'], dtype=object)]

short_description:
['Coyote: An Old West Vignette is a tiny point-and-click adventure game about a gruff marshall on the trail o

## 4. Create feature sets

We prepare the numerical, categorical, list and text features separately. The M4 features will be our baseline, and we will add new features in the following experiments.

In [23]:
# M4 baseline features
baseline_features = [
    "price",
    "windows",
    "mac",
    "linux",
    "required_age",
    "categories_count",
    "genres_count"
]

# Additional numerical features
numeric_features = [
    "price",
    "required_age"
]

# Boolean platform features
boolean_features = [
    "windows",
    "mac",
    "linux"
]

# Categorical feature
categorical_features = [
    "release_month"
]

# Text feature
text_feature = "short_description"

# Multi-value features
list_features = [
    "categories",
    "genres"
]

# Create count features for lists
for df in [train_df, val_df, test_df]:
    df["categories_count"] = df["categories"].apply(len)
    df["genres_count"] = df["genres"].apply(len)

    # Extract release month
    df["release_month"] = df["release_date"].dt.month.astype(str)

print("Baseline features:", baseline_features)
print("Numeric features:", numeric_features)
print("Boolean features:", boolean_features)
print("Categorical features:", categorical_features)
print("Text feature:", text_feature)
print("List features:", list_features)

Baseline features: ['price', 'windows', 'mac', 'linux', 'required_age', 'categories_count', 'genres_count']
Numeric features: ['price', 'required_age']
Boolean features: ['windows', 'mac', 'linux']
Categorical features: ['release_month']
Text feature: short_description
List features: ['categories', 'genres']


## 5. Experiment 1: Standardization

We use StandardScaler to standardize the numerical features. The scaler is fitted using only the training data to prevent data leakage. We then train Logistic Regression and compare its F1-score with the M4 model.

In [24]:
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

# Prepare the baseline feature datasets
X_train = train_df[baseline_features]
X_val = val_df[baseline_features]

y_train = train_df["hit"]
y_val = val_df["hit"]

# Standardize numerical features
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(
    X_train[numeric_features]
)

X_val_scaled[numeric_features] = scaler.transform(
    X_val[numeric_features]
)

# Train Logistic Regression
standardized_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

standardized_model.fit(X_train_scaled, y_train)

# Predict validation data
standardized_predictions = standardized_model.predict(X_val_scaled)

# Calculate F1-score
standardized_f1 = f1_score(
    y_val,
    standardized_predictions
)

print("Standardized Logistic Regression")
print("--------------------------------")
print("F1-score:", round(standardized_f1, 3))

Standardized Logistic Regression
--------------------------------
F1-score: 0.232


## 6. Experiment 2: One-hot encoding

We use OneHotEncoder to convert the release month into numerical binary columns. This allows Logistic Regression to use the release month as a categorical feature.

In [25]:
from sklearn.preprocessing import OneHotEncoder

# Create the encoder
month_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit only on training data
train_month = month_encoder.fit_transform(
    train_df[["release_month"]]
)

val_month = month_encoder.transform(
    val_df[["release_month"]]
)

# Convert encoded data into DataFrames
month_columns = month_encoder.get_feature_names_out(
    ["release_month"]
)

train_month_df = pd.DataFrame(
    train_month,
    columns=month_columns,
    index=train_df.index
)

val_month_df = pd.DataFrame(
    val_month,
    columns=month_columns,
    index=val_df.index
)

# Add encoded month columns to baseline features
X_train_month = pd.concat(
    [X_train_scaled, train_month_df], axis=1
)

X_val_month = pd.concat(
    [X_val_scaled, val_month_df], axis=1
)

# Train Logistic Regression
month_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

month_model.fit(X_train_month, y_train)

# Evaluate
month_predictions = month_model.predict(X_val_month)

month_f1 = f1_score(y_val, month_predictions)

print("One-hot Encoding Results")
print("------------------------")
print("Number of month columns:", len(month_columns))
print("F1-score:", round(month_f1, 3))

One-hot Encoding Results
------------------------
Number of month columns: 12
F1-score: 0.241


## 7. Experiment 3: Text features with TF-IDF

Before using game descriptions, we inspect the text for words that might reveal information about a game's success after release. Such information could cause data leakage.

In [26]:
# Search for potentially post-release wording
suspicious_terms = [
    "award-winning",
    "award winning",
    "acclaimed",
    "bestselling",
    "best-selling",
    "players love",
    "fan favorite",
    "highly rated"
]

for term in suspicious_terms:
    matches = train_df[
        train_df["short_description"].str.contains(
            term,
            case=False,
            na=False,
            regex=False
        )
    ]

    print(f"{term}: {len(matches)} matches")

    if len(matches) > 0:
        display(matches[["name", "short_description"]].head(3))

award-winning: 209 matches


,name,short_description
140,The Interactive Adventures of Dog Mendonça & P...,Based on the award-winning graphic novel serie...
355,LostWinds 2: Winter of the Melodias,"With amazing new abilities, re-visit a land fu..."
616,Age of Wonders III,Age of Wonders III is the long anticipated seq...


award winning: 60 matches


,name,short_description
805,Sentinel 3: Homeworld,The biggest installment of the award winning S...
1526,Arashi: Castles of Sin - Final Cut,Become one with the shadows with the award win...
1900,A Case of the Crabs: Rehash,A complete remake/remaster/rehash of the award...


acclaimed: 139 matches


,name,short_description
187,Ancestors Legacy Free Peasant Edition,Ancestors Legacy Free Peasant Edition represen...
1035,Hearts of Iron III: Semper Fi,It’s time to dust off the field marshal's bato...
1118,URU: Complete Chronicles,The critically acclaimed adventure of Uru: Age...


... (6 outputs left out)

fan favorite: 3 matches


,name,short_description
36288,Micro Tanks,Micro Tanks is that fan favorite artillery tan...
42919,SHOCK TROOPERS,"“SHOCK TROOPERS ”, the fan favorite 2D top-dow..."
43465,Risk of Rain Returns,"Carefully designed, beautifully remastered, an..."


highly rated: 2 matches


,name,short_description
51494,CLANNAD,From the world renowned visual novel studio Ke...
55327,Harmonia,"From renowned visual novel studio Key, which b..."


## 8.1 Cleaning potentially misleading text

We remove selected phrases related to awards, critical acclaim and popularity. We then compare the TF-IDF model using the original descriptions with a model using cleaned descriptions.

In [27]:
import re

# Phrases that may reveal post-release reputation
suspicious_patterns = [
    r"award[- ]winning",
    r"acclaimed",
    r"bestselling",
    r"best-selling",
    r"players love",
    r"fan favorite",
    r"highly rated"
]

def clean_description(text):
    for pattern in suspicious_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    return text

# Create cleaned descriptions
train_df["clean_description"] = train_df["short_description"].apply(clean_description)
val_df["clean_description"] = val_df["short_description"].apply(clean_description)

print("Original description:")
print(train_df["short_description"].iloc[0])

print("\nCleaned description:")
print(train_df["clean_description"].iloc[0])

Original description:
Coyote: An Old West Vignette is a tiny point-and-click adventure game about a gruff marshall on the trail of a deadly outlaw.

Cleaned description:
Coyote: An Old West Vignette is a tiny point-and-click adventure game about a gruff marshall on the trail of a deadly outlaw.


## 8.2 TF-IDF model comparison

We use TF-IDF to convert game descriptions into numerical features. We compare the original descriptions with cleaned descriptions to see whether potentially misleading wording affects model performance.

In [28]:
# Create TF-IDF vectorizers
tfidf_original = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=3
)

tfidf_cleaned = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=3
)

# Fit on training descriptions only
train_text_original = tfidf_original.fit_transform(
    train_df["short_description"]
)
val_text_original = tfidf_original.transform(
    val_df["short_description"]
)

train_text_cleaned = tfidf_cleaned.fit_transform(
    train_df["clean_description"]
)
val_text_cleaned = tfidf_cleaned.transform(
    val_df["clean_description"]
)

# Combine TF-IDF with baseline and release-month features
from scipy.sparse import hstack, csr_matrix

train_base_sparse = csr_matrix(
    X_train_month.to_numpy(dtype=np.float64)
)

val_base_sparse = csr_matrix(
    X_val_month.to_numpy(dtype=np.float64)
)

X_train_original_tfidf = hstack([
    train_base_sparse,
    train_text_original
])

X_val_original_tfidf = hstack([
    val_base_sparse,
    val_text_original
])

X_train_cleaned_tfidf = hstack([
    train_base_sparse,
    train_text_cleaned
])

X_val_cleaned_tfidf = hstack([
    val_base_sparse,
    val_text_cleaned
])

# Train original text model
tfidf_original_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

tfidf_original_model.fit(X_train_original_tfidf, y_train)

original_predictions = tfidf_original_model.predict(
    X_val_original_tfidf
)

original_f1 = f1_score(y_val, original_predictions)

# Train cleaned text model
tfidf_cleaned_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

tfidf_cleaned_model.fit(X_train_cleaned_tfidf, y_train)

cleaned_predictions = tfidf_cleaned_model.predict(
    X_val_cleaned_tfidf
)

cleaned_f1 = f1_score(y_val, cleaned_predictions)

# Display comparison
print("TF-IDF Model Comparison")
print("-----------------------")
print("Original descriptions:", round(original_f1, 3))
print("Cleaned descriptions: ", round(cleaned_f1, 3))
print("Release month only:   ", round(month_f1, 3))

TF-IDF Model Comparison
-----------------------
Original descriptions: 0.392
Cleaned descriptions:  0.393
Release month only:    0.241


## 9. Experiment 4: Multi-hot encoding

Genres and categories can contain multiple values for each game. We use MultiLabelBinarizer to convert these lists into binary features, where 1 means that a game belongs to a particular genre or category.

In [29]:
from sklearn.preprocessing import MultiLabelBinarizer

# Create encoders
genre_encoder = MultiLabelBinarizer()
category_encoder = MultiLabelBinarizer()

# Fit on training data only
train_genres = genre_encoder.fit_transform(train_df["genres"])
val_genres = genre_encoder.transform(val_df["genres"])

train_categories = category_encoder.fit_transform(train_df["categories"])
val_categories = category_encoder.transform(val_df["categories"])

# Convert to sparse matrices
train_genres_sparse = csr_matrix(train_genres)
val_genres_sparse = csr_matrix(val_genres)

train_categories_sparse = csr_matrix(train_categories)
val_categories_sparse = csr_matrix(val_categories)

# Combine genre and category features
train_lists_sparse = hstack([
    train_genres_sparse,
    train_categories_sparse
])

val_lists_sparse = hstack([
    val_genres_sparse,
    val_categories_sparse
])

print("Number of genre features:", len(genre_encoder.classes_))
print("Number of category features:", len(category_encoder.classes_))
print("Total multi-hot features:", train_lists_sparse.shape[1])

Number of genre features: 28
Number of category features: 56
Total multi-hot features: 84


c:\Users\kinga\anaconda3\envs\machinelearning\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['Mods', 'Mods (require HL2)'] will be ignored
  warnings.warn(


## 9.1 Combining all feature types

We combine numerical, categorical, text and multi-value features into one dataset. We then train Logistic Regression to evaluate whether using all feature types improves prediction performance.

In [30]:
# Combine all training features
X_train_all = hstack([
    train_base_sparse,
    train_text_cleaned,
    train_lists_sparse
])

# Combine all validation features
X_val_all = hstack([
    val_base_sparse,
    val_text_cleaned,
    val_lists_sparse
])

# Train Logistic Regression
all_features_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

all_features_model.fit(X_train_all, y_train)

# Predict validation data
all_features_predictions = all_features_model.predict(X_val_all)

# Calculate F1-score
all_features_f1 = f1_score(
    y_val,
    all_features_predictions
)

print("Combined Feature Results")
print("------------------------")
print("Validation F1-score:", round(all_features_f1, 3))
print("Total features:", X_train_all.shape[1])

Combined Feature Results
------------------------
Validation F1-score: 0.489
Total features: 5103


## 9.2 Comparison of feature experiments

We compare the validation F1-scores from the different experiments to identify which feature combinations provide the best performance.

In [31]:
# Compare validation F1-scores
results = pd.DataFrame({
    "Model": [
        "M4 Baseline",
        "Release Month",
        "Cleaned TF-IDF + Release Month",
        "All Combined Features"
    ],
    "Validation F1": [
        0.232,
        month_f1,
        cleaned_f1,
        all_features_f1
    ]
})

display(results.sort_values(
    by="Validation F1",
    ascending=False
).reset_index(drop=True))

,Model,Validation F1
0,All Combined Features,0.488763
1,Cleaned TF-IDF + Release Month,0.393057
2,Release Month,0.241220
3,M4 Baseline,0.232000


## 10. Experiment 5: Building a preprocessing pipeline

We use ColumnTransformer to combine all preprocessing steps into one structure. The pipeline handles numerical scaling, one-hot encoding, text vectorization and multi-hot encoding. This makes it easier to train the model and use it later with new game data.

In [32]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer
from scipy.sparse import hstack, csr_matrix


# Custom transformer for multi-value features
class MultiColumnLabelBinarizer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=object)

        self.genre_encoder = MultiLabelBinarizer(sparse_output=True)
        self.category_encoder = MultiLabelBinarizer(sparse_output=True)

        self.genre_encoder.fit(X[:, 0])
        self.category_encoder.fit(X[:, 1])

        return self

    def transform(self, X):
        X = np.asarray(X, dtype=object)

        genres = self.genre_encoder.transform(X[:, 0])
        categories = self.category_encoder.transform(X[:, 1])

        return hstack([genres, categories], format="csr")


# Clean text descriptions before TF-IDF
def clean_text_column(texts):
    return [
        clean_description(str(text)) if pd.notna(text) else ""
        for text in texts
    ]


# Text preprocessing pipeline
text_pipeline = Pipeline([
    ("clean", FunctionTransformer(
        clean_text_column,
        validate=False
    )),
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        stop_words="english",
        min_df=3
    ))
])


# Combine all preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),

        ("boolean", "passthrough", boolean_features),

        ("counts", "passthrough", [
            "categories_count",
            "genres_count"
        ]),

        ("month", OneHotEncoder(handle_unknown="ignore"),
         categorical_features),

        ("text", text_pipeline, text_feature),

        ("lists", MultiColumnLabelBinarizer(), list_features)
    ],
    remainder="drop"
)


# Complete machine learning pipeline
m5_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


# Train using training data only
m5_pipeline.fit(train_df, y_train)


# Predict validation data
pipeline_predictions = m5_pipeline.predict(val_df)

pipeline_f1 = f1_score(
    y_val,
    pipeline_predictions
)


print("M5 Pipeline Results")
print("-------------------")
print("Validation F1-score:", round(pipeline_f1, 3))

M5 Pipeline Results
-------------------
Validation F1-score: 0.491


c:\Users\kinga\anaconda3\envs\machinelearning\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['Mods', 'Mods (require HL2)'] will be ignored
  warnings.warn(


## 11. Experiment 6: Five-fold cross-validation

We use five-fold cross-validation to compare the M4 baseline model with the M5 pipeline. This gives a better indication of how consistently each model performs across different subsets of the training data.

In [33]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Define 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# M4 baseline model
m4_cv_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Use the original M4 features
X_train_m4 = train_df[baseline_features]

# M5 pipeline is already defined in Cell 28
# Cross-validation uses training data only

# Evaluate M4 baseline
m4_cv_scores = cross_val_score(
    m4_cv_model,
    X_train_m4,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

# Evaluate M5 pipeline
m5_cv_scores = cross_val_score(
    m5_pipeline,
    train_df,
    y_train,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

# Print results
print("5-Fold Cross-Validation Results")
print("--------------------------------")

print("M4 Baseline:")
print("Fold scores:", np.round(m4_cv_scores, 3))
print("Mean F1:", round(m4_cv_scores.mean(), 3))
print("Standard deviation:", round(m4_cv_scores.std(), 3))

print("\nM5 Pipeline:")
print("Fold scores:", np.round(m5_cv_scores, 3))
print("Mean F1:", round(m5_cv_scores.mean(), 3))
print("Standard deviation:", round(m5_cv_scores.std(), 3))

5-Fold Cross-Validation Results
--------------------------------
M4 Baseline:
Fold scores: [0.244 0.233 0.223 0.225 0.23 ]
Mean F1: 0.231
Standard deviation: 0.008

M5 Pipeline:
Fold scores: [0.491 0.484 0.479 0.467 0.477]
Mean F1: 0.48
Standard deviation: 0.008


### 11.1 Cross-validation results and conclusion

The five-fold cross-validation results show that the M5 pipeline performs considerably better than the M4 baseline.

The M4 baseline achieved a mean F1-score of 0.231 with a standard deviation of 0.008. The M5 pipeline achieved a mean F1-score of 0.480 with the same standard deviation of 0.008.

Since the M5 pipeline performs better across all five folds, the improvement appears consistent. Therefore, the M5 pipeline is selected for further evaluation.

## 12. Experiment 7: Selecting a recommendation threshold

The default classification threshold is 0.5. However, Pixel Harbor has limited capacity to review recommended games. We use validation data to select a threshold that limits recommendations to approximately 10% of games, following the same approach as M4.

In [34]:
# Get hit probabilities from the M5 pipeline
m5_val_probabilities = m5_pipeline.predict_proba(val_df)[:, 1]

# Set review capacity to 10%
capacity = 0.10
max_recommendations = int(len(val_df) * capacity)

# Find threshold that recommends approximately 10% of games
m5_threshold = np.sort(m5_val_probabilities)[-max_recommendations]

# Make predictions using the selected threshold
m5_val_predictions = (
    m5_val_probabilities >= m5_threshold
).astype(int)

# Evaluate results
print("M5 Threshold Selection")
print("----------------------")
print("Selected threshold:", round(m5_threshold, 4))
print("Recommended games:", m5_val_predictions.sum())
print("Review capacity:", f"{capacity:.0%}")
print("Precision:", round(precision_score(y_val, m5_val_predictions), 3))
print("Recall:", round(recall_score(y_val, m5_val_predictions), 3))
print("F1-score:", round(f1_score(y_val, m5_val_predictions), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, m5_val_predictions))

M5 Threshold Selection
----------------------
Selected threshold: 0.6016
Recommended games: 1950
Review capacity: 10%
Precision: 0.734
Recall: 0.292
F1-score: 0.418

Confusion Matrix:
[[14080   518]
 [ 3476  1432]]


c:\Users\kinga\anaconda3\envs\machinelearning\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['Mods', 'Mods (require HL2)'] will be ignored
  warnings.warn(


### 12.1 Threshold selection results

The M5 pipeline achieved better results than M4 when using the same 10% review capacity.

At a threshold of 0.6016, M5 recommended 1,950 games. It achieved a precision of 0.734, recall of 0.292 and F1-score of 0.418.

Compared with M4, M5 improved precision, recall and F1-score while keeping the same number of recommendations. Therefore, M5 is selected as the preferred model.

## 13. Final evaluation of the M5 model

We evaluate the selected M5 pipeline on the test dataset using the threshold chosen from the validation dataset. The test dataset has not been used for model training or threshold selection, so it provides an independent evaluation of the model.

In [35]:
# Use the threshold selected from validation data
final_m5_threshold = m5_threshold

# Get hit probabilities for the test dataset
m5_test_probabilities = m5_pipeline.predict_proba(test_df)[:, 1]

# Apply the selected threshold
m5_test_predictions = (
    m5_test_probabilities >= final_m5_threshold
).astype(int)

# Calculate evaluation metrics
m5_test_precision = precision_score(
    test_df["hit"],
    m5_test_predictions
)

m5_test_recall = recall_score(
    test_df["hit"],
    m5_test_predictions
)

m5_test_f1 = f1_score(
    test_df["hit"],
    m5_test_predictions
)

# Print results
print("Final M5 Test Results")
print("---------------------")
print("Threshold:", round(final_m5_threshold, 4))
print("Recommended games:", m5_test_predictions.sum())
print("Precision:", round(m5_test_precision, 3))
print("Recall:", round(m5_test_recall, 3))
print("F1-score:", round(m5_test_f1, 3))

print("\nConfusion Matrix:")
print(confusion_matrix(
    test_df["hit"],
    m5_test_predictions
))

Final M5 Test Results
---------------------
Threshold: 0.6016
Recommended games: 1908
Precision: 0.747
Recall: 0.288
F1-score: 0.415

Confusion Matrix:
[[14069   483]
 [ 3529  1425]]


c:\Users\kinga\anaconda3\envs\machinelearning\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['Mods', 'Mods (require HL2)'] will be ignored
  warnings.warn(
c:\Users\kinga\anaconda3\envs\machinelearning\Lib\site-packages\sklearn\preprocessing\_label.py:1016: UserWarning: unknown class(es) ['360 Video', 'Documentary', 'Episodic', 'Short', 'Tutorial'] will be ignored
  warnings.warn(


## 14. M5 Final Conclusion

The M5 pipeline performed considerably better than the M4 baseline.

The five-fold cross-validation results showed a mean F1-score of 0.480 for M5, compared with 0.231 for M4.

Using a recommendation capacity of 10%, the selected threshold was 0.6016. On the test dataset, M5 achieved a precision of 0.747, recall of 0.288 and F1-score of 0.415.

The model recommended 1,908 games out of 19,506 test games. Compared with M4, M5 provides more accurate recommendations while keeping the review capacity approximately the same.

The results indicate that adding game descriptions, genres, categories and release month improves prediction performance. Therefore, the M5 pipeline is selected as the preferred model for further development.